<a href="https://colab.research.google.com/github/RAJAMURUGAN-VS/genai-learning-journey/blob/main/04-rag/03_pdf_rag_assistant_deployment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%%writefile requirements.txt
gradio
langchain
langchain-community
langchain-text-splitters
langchain-huggingface
langchain-chroma
chromadb
pypdf
sentence-transformers
langchain-google-genai
google-genai

Writing requirements.txt


In [ ]:
%%writefile app.py
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain.chat_models import init_chat_model
import gradio as gr

cached_vector_stores = {}


def build_vector_store(pdf_url):

  loader=PyPDFLoader(pdf_url)
  doc=loader.load()

  text_splitter=RecursiveCharacterTextSplitter(
      chunk_size=1000,
      chunk_overlap=200,
      length_function=len,
  )

  all_splits=text_splitter.split_documents(doc)

  embedding_model=HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
  )

  vector_store=Chroma(
    collection_name="research_collection",
    embedding_function=embedding_model,
    persist_directory="./chroma_langchain_db"
  )

  vector_store.add_documents(documents=all_splits)

  return vector_store


def get_vector_store(pdf_url):

  if pdf_url in cached_vector_stores:
    return cached_vector_stores[pdf_url]

  vector_store=build_vector_store(pdf_url)

  cached_vector_stores[pdf_url]=vector_store

  return vector_store


def retrieve_context(vector_store, query, k=5):

  retrieved_docs=vector_store.similarity_search(query, k=k)

  docs_content=""
  for doc in retrieved_docs:
    docs_content+=f"Source: {doc.metadata}\n"
    docs_content+=f"Content: {doc.page_content}\n\n"

  return docs_content, retrieved_docs


api_key=os.getenv('GEMINI_API_KEY')
model=init_chat_model(
   "google_genai:gemini-2.5-flash",
   api_key=api_key,
)


def docu_chat(pdf_url, user_query):

  try:
    vector_store=get_vector_store(pdf_url)

    context, source_docs=retrieve_context(vector_store, user_query, k=5)

    system_message=f"""
    You are a helpful chatbot.
    Use only the following pieces of context to answer the question.
    Don't makeup any new information.
    Context: {context}
    """

    messages=[
      {"role": "system", "content": system_message},
      {"role": "user", "content": user_query}
    ]

    response=model.invoke(messages)

    sources = []

    for doc in source_docs:
      page = doc.metadata.get("page", 0) + 1
      sources.append(f"Page {page}")

    return (response.content+"\n\nSources:\n"+ "\n".join(sources))
  except Exception as e:
        return f"❌ Error: {str(e)}"


demo = gr.Interface(
    fn=docu_chat,
    inputs=[
        gr.Textbox(
            label="PDF URL",
            placeholder="https://arxiv.org/pdf/1706.03762"
        ),
        gr.Textbox(
            lines=4,
            label="Question",
            placeholder="Ask a question about the document..."
        )
    ],
    outputs=gr.Textbox(
        lines=10,
        label="Answer"
    ),
    title="PDF RAG Assistant",
    description="Enter a PDF URL and ask questions about the document."
)


demo.launch()

Writing app.py
